# Qwen3-Embedding-8B on Kaggle T4×2 — Production Demo (v1.0.0)

**English:** Recommended role for this notebook: **USE / DEMO**. It runs the canonical 100 000-row × 4096-dim bilingual semantic-search demo on two independent T4 GPUs and records secret-safe evidence.

**Tiếng Việt:** Vai trò khuyến nghị của notebook này: **USE / DEMO**. Notebook chạy bản demo tìm kiếm ngữ nghĩa song ngữ chuẩn hoá 100 000 dòng × 4096 chiều trên hai GPU T4 độc lập và ghi lại bằng chứng an toàn bí mật.

```text
Recommended / Khuyến nghị: this notebook = USE / DEMO
Qualification / Xác minh: kaggle/demo.ipynb in a different fresh session
GPU: T4×2
Internet: ON for Git clone
Kaggle inputs: attach model + dataset
Run: Restart Session → Run All
```

**English:** The qualification counterpart that proves the frozen runtime envelope lives in `kaggle/demo.ipynb` (`VERIFY / QUALIFICATION`) and must run in a separate fresh Kaggle session. This notebook never reuses `/kaggle/working` state from another runtime.

**Tiếng Việt:** Bản xác minh định lượng (runtime envelope) nằm ở `kaggle/demo.ipynb` (`VERIFY / QUALIFICATION`) và phải chạy trong một phiên Kaggle mới hoàn toàn khác. Notebook này không tái sử dụng trạng thái `/kaggle/working` từ runtime khác.

## Stage 0 — exact-ref clone / hard refresh

**English:** This notebook clones the official public repository at an exact Git ref (`v1.0.0` by default, or a maintainer staging SHA via `REPO_REF_OVERRIDE`) and enforces a clean tree. It never silently falls back to `main`. This exact-ref flow is the same contract used by the qualification notebook.

**Tiếng Việt:** Notebook sao chép kho chứa công khai theo một Git ref chính xác (mặc định `v1.0.0`, hoặc staging SHA của người duy trì qua `REPO_REF_OVERRIDE`) và yêu cầu cây làm việc sạch. Không bao giờ âm thầm quay về nhánh `main`. Luồng exact-ref này giống hệt contract của notebook xác minh.

```python
REPO_URL = "https://github.com/dangkhoa2016/Qwen3-Embedding-8B-On-T4x2-GPU.git"
REPO_REF = "v1.0.0"
REPO_REF_OVERRIDE = ""  # maintainer-only: exact staging SHA during prepublication acceptance
CHECKOUT = Path("/kaggle/working/Qwen3-Embedding-8B-On-T4x2-GPU")
SELECTED_REF = REPO_REF_OVERRIDE.strip() or REPO_REF
```

In [ ]:
import os, sys, json, subprocess, secrets, shutil, hashlib, time, math, platform, re, signal, importlib
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor

REPO_URL = "https://github.com/dangkhoa2016/Qwen3-Embedding-8B-On-T4x2-GPU.git"
REPO_REF = "v1.0.0"
REPO_REF_OVERRIDE = ""  # maintainer-only: exact staging SHA during prepublication acceptance
CHECKOUT = Path("/kaggle/working/Qwen3-Embedding-8B-On-T4x2-GPU")
SELECTED_REF = REPO_REF_OVERRIDE.strip() or REPO_REF

EXPECTED_MODEL_FP = "b41f597aceb7c52762877f2db5bf75504ee243e7382c21ef1608e07b41eb60da"
EXPECTED_DATASET_SHA = "1f69b4b64a0aecf41505aef3cf6fa5c5ce19e4a733d198765ee6f431beb711e9"

WORK = Path("/kaggle/working/qwen3-embedding-8b-production-demo")
INDEX_DIR = WORK / "index"
EVIDENCE_DIR = WORK / f"evidence-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
LOG_DIR = WORK / "logs"
SECRET_DIR = WORK / "secrets"
for _d in (EVIDENCE_DIR, LOG_DIR, SECRET_DIR):
    _d.mkdir(parents=True, exist_ok=True)
LEGACY_PROJECT_DIR_HINT = Path('/kaggle/working/qwen3-embedding-8b-t4x2')

def run(*args, cwd=None, capture=False, env=None):
    return subprocess.run(
        list(args), cwd=cwd, check=True, text=True,
        capture_output=capture, env=env,
    )

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, ensure_ascii=False) + "\n", encoding="utf-8")


def bootstrap_and_adopt_environment(project_root):
    shell_command = (
        "source kaggle/bootstrap.sh 1>&2\n"
        "env -0\n"
    )

    proc = subprocess.run(
        ["bash", "-c", shell_command],
        cwd=str(project_root),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    bootstrap_log = proc.stderr.decode("utf-8", errors="replace")

    if bootstrap_log:
        print(bootstrap_log[-6000:])

    if proc.returncode != 0:
        raise RuntimeError(f"bootstrap failed with rc={proc.returncode}")

    exported = {}

    for item in proc.stdout.split(b"\0"):
        if not item:
            continue

        key, sep, value = item.partition(b"=")

        if not sep:
            continue

        exported[
            key.decode("utf-8", errors="strict")
        ] = value.decode("utf-8", errors="surrogateescape")

    for key in ("PATH", "PYTHONPATH"):
        if not exported.get(key):
            raise RuntimeError(f"bootstrap environment missing {key}")

    for key in ("PATH", "PYTHONPATH", "LD_LIBRARY_PATH"):
        if key in exported:
            os.environ[key] = exported[key]

    return {
        key: os.environ[key]
        for key in ("PATH", "PYTHONPATH", "LD_LIBRARY_PATH")
        if key in os.environ
    }


In [ ]:
if (CHECKOUT / '.git').is_dir():
    actual_origin = run('git', 'remote', 'get-url', 'origin', cwd=CHECKOUT, capture=True).stdout.strip()
    if (actual_origin.rstrip('/') != REPO_URL.rstrip('/').removesuffix('.git')
            and actual_origin.rstrip('/') != REPO_URL.rstrip('/')):
        run('git', 'remote', 'set-url', 'origin', REPO_URL, cwd=CHECKOUT)
    run('git', 'fetch', '--prune', 'origin',
        '+refs/heads/*:refs/remotes/origin/*', '+refs/tags/*:refs/tags/*', cwd=CHECKOUT)
else:
    if CHECKOUT.exists():
        shutil.rmtree(CHECKOUT)
    run('git', 'clone', '--no-checkout', REPO_URL, str(CHECKOUT))
    run('git', 'fetch', '--prune', 'origin',
        '+refs/heads/*:refs/remotes/origin/*', '+refs/tags/*:refs/tags/*', cwd=CHECKOUT)

run('git', 'checkout', '--detach', '--force', SELECTED_REF, cwd=CHECKOUT)
head = run("git", "rev-parse", "HEAD", cwd=CHECKOUT, capture=True).stdout.strip()
status = run("git", "status", "--porcelain", cwd=CHECKOUT, capture=True).stdout
if status:
    raise RuntimeError(f"checkout is dirty: {status}")
os.chdir(CHECKOUT)
sys.path.insert(0, str(CHECKOUT))
PROJECT_ROOT = CHECKOUT
print('SELECTED_REF=', SELECTED_REF)
print('GIT_HEAD=', head)
BOOTSTRAP_ENV = bootstrap_and_adopt_environment(PROJECT_ROOT)
print('BOOTSTRAP_ENV_ADOPTED=', sorted(BOOTSTRAP_ENV.keys()))
for _rel in ('app/main.py', 'kaggle/bootstrap.sh', 'kaggle/run-demo.sh', 'scripts/build-index.py'):
    if not (CHECKOUT / _rel).is_file():
        raise RuntimeError(f'missing required file in checkout: {_rel}')
print('CHECKOUT_FILES=OK')
write_json(EVIDENCE_DIR / 'git-identity.txt', {
    'selected_ref': SELECTED_REF,
    'head': head,
    'status': 'clean',
})
print('GIT_IDENTITY=OK')

## Stage 1 — runtime / T4×2 preflight

**English:** Records UTC time, Python/platform/kernel, the NVIDIA GPU inventory (nvidia-smi), Torch version/file/CUDA availability and device names, then disk usage. A hard gate refuses to continue with fewer than two CUDA devices — this demo requires two T4 GPUs (T4×2), one replica per T4, and never downgrades to a single GPU.

**Tiếng Việt:** Ghi lại thời gian UTC, Python/nền tảng/kernel, thông tin GPU NVIDIA (nvidia-smi), phiên bản/tệp Torch, khả năng CUDA và tên thiết bị, rồi dung lượng ổ đĩa. Cổng chặn cứng từ chối tiếp tục nếu có ít hơn hai thiết bị CUDA — demo này yêu cầu hai GPU T4 (T4×2), mỗi T4 một replica độc lập, và không bao giờ hạ xuống một GPU.

In [ ]:
UTC_TIMESTAMP = datetime.now(timezone.utc).isoformat()
print('UTC_TIMESTAMP=', UTC_TIMESTAMP)
print('PYTHON=', platform.python_version())
print('PLATFORM=', platform.platform())
print('KERNEL=', platform.uname().release)
write_json(EVIDENCE_DIR / 'environment.txt', {
    'utc_timestamp': UTC_TIMESTAMP,
    'python': platform.python_version(),
    'platform': platform.platform(),
    'kernel': platform.uname().release,
})

def _nvidia_smi_paths():
    return ['nvidia-smi', '/opt/bin/nvidia-smi', '/opt/nvidia/bin/nvidia-smi']

nvidia_found = None
for cand in _nvidia_smi_paths():
    if shutil.which(cand) is not None or Path(cand).is_file():
        nvidia_found = cand
        break
if nvidia_found is None:
    print('NVIDIA_SMI=missing (bootstrap will surface it as a hard error if unavailable)')
else:
    gpu_inventory = subprocess.run(
        [nvidia_found, '--query-gpu=index,name,memory.total', '--format=csv'],
        text=True, capture_output=True,
    )
    print(gpu_inventory.stdout)
    if gpu_inventory.returncode == 0:
        (EVIDENCE_DIR / 'nvidia-smi-before.txt').write_text(
            gpu_inventory.stdout + gpu_inventory.stderr, encoding='utf-8')
    else:
        print(gpu_inventory.stderr)

import torch
print('TORCH=', torch.__version__)
print('TORCH_FILE=', torch.__file__)
print('CUDA_AVAILABLE=', torch.cuda.is_available())
print('DEVICE_COUNT=', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'DEVICE_{i}=', torch.cuda.get_device_name(i))
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable')
if torch.cuda.device_count() < 2:
    raise RuntimeError(f'Need T4×2 / at least 2 CUDA GPUs, found {torch.cuda.device_count()}')
print('T4X2_GATE=PASS')
print('--- disk usage ---')
subprocess.run(['df', '-h', '/kaggle/working', '/kaggle/input'], text=True)

## Stage 2 — bootstrap state confirmation

**English:** Bootstrap already ran in Stage 0 immediately after exact checkout. Its PATH/PYTHONPATH/LD_LIBRARY_PATH exports were adopted into the notebook process before the Stage-1 GPU preflight. This stage records/confirms that bootstrap state rather than running bootstrap a second time.

**Tiếng Việt:** Bootstrap đã được chạy ở Stage 0 ngay sau khi checkout đúng ref. PATH/PYTHONPATH/LD_LIBRARY_PATH đã được nhập vào môi trường của notebook trước phần GPU preflight ở Stage 1. Stage này chỉ ghi nhận/xác nhận trạng thái đó, không chạy bootstrap lần hai.

In [ ]:
(LOG_DIR / 'bootstrap.log').write_text(
    'bootstrap executed and env adopted in Stage 0 via BOOTSTRAP_ENV\n',
    encoding='utf-8',
)
print('BOOTSTRAP_ENV_KEYS=', sorted(BOOTSTRAP_ENV.keys()))
print('BOOTSTRAP=ALREADY_ADOPTED_IN_STAGE_0')


## Stage 3 — model + dataset resolution

**English:** Uses the repository's offline resolvers to locate the attached Kaggle Model and Kaggle Dataset under `/kaggle/input`, then requires the exact expected model fingerprint and dataset SHA-256. Only safe source/fingerprint/license/row metadata is printed or retained — credentials are never printed.

**Tiếng Việt:** Dùng bộ phân giải offline của kho để tìm Kaggle Model và Kaggle Dataset đã đính kèm dưới `/kaggle/input`, sau đó yêu cầu đúng model fingerprint và SHA-256 dữ liệu dự kiến. Chỉ in/lưu metadata nguồn/fingerprint/giấy phép/số dòng an toàn — không bao giờ in thông tin đăng nhập.

In [ ]:
from app.model_resolver import resolve_kaggle_model_dir, model_fingerprint
from app.dataset_resolver import resolve_kaggle_dataset_dir, validate_dataset_dir

MODEL_ROOT = Path('/kaggle/input')
model_dir = resolve_kaggle_model_dir(MODEL_ROOT, os.environ.get('KAGGLE_MODEL_DIR'))
fp = model_fingerprint(model_dir)
assert fp == EXPECTED_MODEL_FP, f'unexpected model fingerprint: {fp}'
print('MODEL_SOURCE=kaggle_input')
print('MODEL_FINGERPRINT=', fp)
print('MODEL_DIR=resolved under /kaggle/input (path withheld from evidence)')

dataset_dir = resolve_kaggle_dataset_dir(MODEL_ROOT, os.environ.get('KAGGLE_DATASET_DIR'))
dmanifest = validate_dataset_dir(dataset_dir)
assert dmanifest['canonical_sha256'] == EXPECTED_DATASET_SHA, 'dataset SHA mismatch'
print('DATASET_SOURCE=', dmanifest.get('source'))
print('DATASET_LICENSE=', dmanifest.get('source_license'))
print('DATASET_ROWS=', dmanifest.get('row_count'))
print('DATASET_SHA256=', dmanifest['canonical_sha256'])
write_json(EVIDENCE_DIR / 'input-preflight.txt', {
    'model_source': 'kaggle_input',
    'model_fingerprint': fp,
    'dataset_source': dmanifest.get('source'),
    'dataset_license': dmanifest.get('source_license'),
    'dataset_row_count': dmanifest.get('row_count'),
    'dataset_sha256': dmanifest['canonical_sha256'],
})
print('INPUT_PREFLIGHT=OK')

## Stage 4 — canonical 100K index

**English:** Builds or reuses the canonical 100 000-row × 4096-dim index. The build command deliberately omits `--limit`: a reduced corpus is forbidden in this production path. After the build the notebook fails unless metadata is exactly 100000 × 4096 with the expected model fingerprint and dataset SHA-256.

**Tiếng Việt:** Xây dựng hoặc tái sử dụng index chuẩn 100 000 dòng × 4096 chiều. Lệnh xây dựng cố tình không dùng `--limit`: cấm giảm bớt corpus trong đường dẫn production này. Sau khi xây, notebook sẽ dừng nếu metadata không chính xác 100000 × 4096 với đúng model fingerprint và SHA-256 dữ liệu dự kiến.

In [ ]:
expected_meta = {
    'model_fingerprint': EXPECTED_MODEL_FP,
    'dimensions': 4096,
    'row_count': 100000,
    'dataset_sha256': EXPECTED_DATASET_SHA,
}
meta_path = INDEX_DIR / 'index-metadata.json'
reuse = False
if meta_path.is_file():
    try:
        existing = json.loads(meta_path.read_text(encoding='utf-8'))
        reuse = (
            (INDEX_DIR / 'vectors.npy').is_file()
            and (INDEX_DIR / 'rows.json').is_file()
            and all(existing.get(k) == v for k, v in expected_meta.items())
        )
    except (OSError, json.JSONDecodeError):
        reuse = False
if reuse:
    print('INDEX_REUSE=YES')
else:
    if INDEX_DIR.is_dir():
        shutil.rmtree(INDEX_DIR)
    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    index_build = subprocess.run(
        [sys.executable, 'scripts/build-index.py',
         '--output-dir', str(INDEX_DIR), '--dimensions', '4096'],
        text=True, capture_output=True,
    )
    (LOG_DIR / 'index-build.log').write_text(
        index_build.stdout + index_build.stderr, encoding='utf-8')
    print(index_build.stdout[-3000:])
    if index_build.returncode != 0:
        print('--- index build stderr (tail) ---')
        print(index_build.stderr[-2000:])
        raise RuntimeError('index build failed')
    print('INDEX_BUILD=OK')
meta = json.loads(meta_path.read_text(encoding='utf-8'))
assert meta['row_count'] == 100000, meta
assert meta['dimensions'] == 4096, meta
assert meta['dataset_sha256'] == EXPECTED_DATASET_SHA, meta
assert meta['model_fingerprint'] == EXPECTED_MODEL_FP, meta
shutil.copyfile(meta_path, EVIDENCE_DIR / 'index-metadata.json')
print('INDEX_METADATA=100000x4096 OK')

## Stage 5 — ephemeral local bearer key

**English:** Creates an ephemeral API key under `/kaggle/working/<work>/secrets/api-key` with mode `0600`. The raw file is never copied into evidence or release artifacts, and the key itself is never printed.

**Tiếng Việt:** Tạo API key tạm thời tại `/kaggle/working/<work>/secrets/api-key` với quyền `0600`. Tệp gốc không bao giờ được sao chép vào evidence hoặc artifact phát hành, và bản thân key không bao giờ được in ra.

In [ ]:
API_KEY_FILE = SECRET_DIR / 'api-key'
if not API_KEY_FILE.exists():
    API_KEY_FILE.write_text(secrets.token_urlsafe(32) + '\n', encoding='utf-8')
    try:
        os.chmod(API_KEY_FILE, 0o600)
    except OSError:
        pass
api_key = API_KEY_FILE.read_text(encoding='utf-8').strip()
assert api_key and len(api_key) >= 40
print('API_KEY_CREATED=YES (value withheld)')

## Stage 6 — start dual-GPU FastAPI service

**English:** Launches the repository's existing `kaggle/run-demo.sh`, which starts a **single Uvicorn parent** (`--workers 1`) on port 8000 plus two spawned model workers (one per T4). Configuration is fixed to `WORKER_COUNT=2`, `MAX_BATCH_ESTIMATED_TOKENS=512`, `EXPOSE_MODE=off`, and the ephemeral `API_KEY`. No second API implementation is started inside notebook cells.

**Tiếng Việt:** Khởi chạy `kaggle/run-demo.sh` có sẵn trong kho, vốn khởi động **một tiến trình cha Uvicorn duy nhất** (`--workers 1`) trên cổng 8000 kèm hai model worker (mỗi T4 một worker). Cấu hình cố định `WORKER_COUNT=2`, `MAX_BATCH_ESTIMATED_TOKENS=512`, `EXPOSE_MODE=off` và `API_KEY` tạm thời. Không có API thứ hai nào được khởi động trong các ô notebook.

In [ ]:
SERVER_LOG = LOG_DIR / 'server.log'
cmd_env = os.environ.copy()
cmd_env.update({
    'WORKING_DIR': str(WORK),
    'WORKER_COUNT': '2',
    'PORT': '8000',
    'SEARCH_INDEX_DIR': str(INDEX_DIR),
    'EXPOSE_MODE': 'off',
    'MAX_BATCH_ESTIMATED_TOKENS': '512',
    'API_KEY': api_key,
})
SERVER_STARTED_UTC = datetime.now(timezone.utc).isoformat()
SERVER_STARTED_MONO = time.monotonic()
# kaggle/run-demo.sh starts a single Uvicorn parent with --workers 1; do not start a second API here.
with open(SERVER_LOG, 'wb') as server_log_fh:
    server_proc = subprocess.Popen(
        ['bash', 'kaggle/run-demo.sh'], env=cmd_env,
        stdout=server_log_fh, stderr=subprocess.STDOUT,
    )
print('SERVER_PID=', server_proc.pid)
print('SERVER_STARTED_UTC=', SERVER_STARTED_UTC)

## Stage 7 — readiness matrix

**English:** Polls the local service until both workers report ready, then asserts the fail-closed readiness matrix: `GET /health` → `200` without auth, `GET /ready` → `401` without auth, `GET /ready` with a valid Bearer header → `200`. On timeout, only safe diagnostics are printed (server log tail, nvidia-smi, process tree, disk usage, index metadata). The key is never printed.

**Tiếng Việt:** Chờ dịch vụ cục bộ sẵn sàng khi cả hai worker đã load xong, rồi kiểm tra ma trận sẵn sàng fail-closed: `GET /health` → `200` không cần xác thực, `GET /ready` → `401` không xác thực, `GET /ready` kèm Bearer hợp lệ → `200`. Khi hết thời gian, chỉ in chẩn đoán an toàn (đuôi log server, nvidia-smi, cây tiến trình, dung lượng ổ, metadata index). Key không bao giờ được in.

In [ ]:
import httpx
BASE = "http://127.0.0.1:8000"
AUTH = {"Authorization": f"Bearer {api_key}"}

def _ready_auth(timeout_total=900):
    deadline = time.time() + timeout_total
    while time.time() < deadline:
        try:
            r = httpx.get(BASE + '/ready', headers=AUTH, timeout=10)
            if r.status_code == 200:
                return r
        except Exception:
            pass
        time.sleep(5)
    print('--- server.log tail ---')
    print(SERVER_LOG.read_text(encoding='utf-8', errors='replace')[-4000:])
    subprocess.run(['ps', '-eo', 'pid,ppid,args'], text=True)
    subprocess.run(['df', '-h', '/kaggle/working'], text=True)
    raise RuntimeError('server did not become ready in time')

ready_response = _ready_auth()
matrix = {}
matrix['health_no_auth'] = httpx.get(BASE + '/health', timeout=10).status_code
matrix['ready_no_auth'] = httpx.get(BASE + '/ready', timeout=10).status_code
matrix['ready_valid_auth'] = httpx.get(BASE + '/ready', headers=AUTH, timeout=10).status_code
assert matrix['health_no_auth'] == 200, matrix
assert matrix['ready_no_auth'] == 401, matrix
assert matrix['ready_valid_auth'] == 200, matrix
ready_elapsed = round(time.monotonic() - SERVER_STARTED_MONO, 3)
write_json(EVIDENCE_DIR / 'startup-timing.json', {
    'started_utc': SERVER_STARTED_UTC,
    'ready_utc': datetime.now(timezone.utc).isoformat(),
    'ready_elapsed_seconds': ready_elapsed,
})
write_json(EVIDENCE_DIR / 'ready.json', {
    'matrix': matrix,
    'utc': datetime.now(timezone.utc).isoformat(),
    'interpretation': '401 is the fail-closed default; 200 requires a valid ephemeral Bearer key only.',
})
print('READY_MATRIX=', matrix)
print('READY_ELAPSED_SECONDS=', ready_elapsed)